# Driver Audit 
This is a driver audit file to inspect the accuracy, precision, and recall of each driver in one race.

In [ ]:
#importing packages
import pandas as pd
from pathlib import Path

In [ ]:
# Directory and File Setup
BASE_DIR = Path.cwd().parent
AUDIT_CSV = BASE_DIR / "outputs" / "race_demo_2024_Abu_Dhabi_Grand_Prix.csv"

In [ ]:
print("Precision: Ratio Between True Positives and All Positives")
print("How many times did the driver acutally need to pit when the model said 'pit'?\n")
print("Recall: How well the model finds positives.")
print("How many times did the model say 'pit' when the driver needed to pit?\n")

print("Consequences of bad recall: Missing an optimal pit time, stays on the track longer")
print("Consequences of bad precision: When the model tells the driver to pit, it is usually a false alarm, forcing an unnecessary stop that loses them track position.\n")

In [ ]:
def driver_audit(csv_file):
  #load data
  df = pd.read_csv(csv_file)

  driver_stats = df.groupby('driverCode').apply(lambda g: pd.Series({
    'total_count': len(g),
    'TP': (g['Evaluation'] == 'MATCH (Correct Pit Call)').sum(),
    'FP': (g['Evaluation'] == 'FALSE ALARM (Early Call)').sum(),
    'TN': (g['Evaluation'] == 'MATCH (Stay Out)').sum(),
    'FN': (g['Evaluation'] == 'MISSED PIT STOP').sum(),
  }), include_groups=False).reset_index()

  # Calculate your metrics
  driver_stats['accuracy'] = (driver_stats['TP'] + driver_stats['TN']) / (driver_stats['TP'] + driver_stats['TN'] + driver_stats['FP'] + driver_stats['FN'])
  driver_stats['precision'] = driver_stats['TP'] / (driver_stats['TP'] + driver_stats['FP']).replace(0, 1)
  driver_stats['recall'] = driver_stats['TP'] / (driver_stats['TP'] + driver_stats['FN']).replace(0, 1)


  print(driver_stats[['driverCode', 'total_count', 'accuracy', 'precision', 'recall']])

In [ ]:
driver_audit(AUDIT_CSV)